# Week 3 Day 4 - Lab 1: AgentCore Runtime

## Deploying a small agent to AWS

Today the UnderCover Agent joins the app on AWS. Before Lab 2 deploys the product, this lab starts with a hello world of your own, then puts a small LangGraph agent on AgentCore Runtime with the AgentCore CLI, to show how Runtime works on its own: how a deploy builds and ships the code, what a session is, and how a graph that pauses in one session carries on in another.

## 7 parts:

PART 1: Hello, AgentCore

PART 2: A LangGraph agent's project

PART 3: The agent on your laptop

PART 4: Deploying to Runtime

PART 5: Sessions

PART 6: Pausing in one session and resuming in another

PART 7: Teardown

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">You need the AWS account and the <code>undercover</code> IAM user from Day 1, with <code>OPENROUTER_API_KEY</code> and <code>DATABASE_URL</code> in your <code>.env</code> file, and the AgentCore CLI at the version I recorded this lab on. In a terminal, run:<br/><br/><code>npm install -g @aws/agentcore@1.0.0-rc.5</code><br/><br/>Then <code>agentcore --version</code> should say 1.0.0-rc.5. The CLI's commands are best run in a terminal, so keep one open beside this notebook.<br/><br/>Today you also need the <code>runtime</code> and <code>agentcore</code> folders from <code>setup/setup_week3day4</code>. If you're on my <code>week3day4-start</code> branch, they're already at the root of the repo. Otherwise, move them there. In a terminal in the <code>undercover</code> folder, run:<br/><br/><code>git mv setup/setup_week3day4/runtime runtime</code><br/><code>git mv setup/setup_week3day4/agentcore agentcore</code><br/><br/>Runtime charges only while a session is working, so this lab costs a fraction of a cent on AWS, and the model calls a fraction of a cent on OpenRouter. The first deploy prepares your account for the CLI, which includes a KMS key at $1 a month, about 3 cents a day. It stays for Lab 2, which shows how to remove it. Part 7 deletes the agents.<br/><br/>If you're on the free route, this lab needs AWS, so please watch me. Lab 2 has a guide for running the agent on Render.
            </span>
        </td>
    </tr>
</table>

## Traducción al español

<h2 style="color:#EF476F;">Antes de empezar</h2>

Necesitas la cuenta de AWS y el usuario IAM `undercover` del Día 1, con `OPENROUTER_API_KEY` y `DATABASE_URL` en tu archivo `.env`, y la CLI de AgentCore en la versión con la que registré este laboratorio. En una terminal, ejecuta:

`npm install -g @aws/agentcore@1.0.0-rc.5`

Luego `agentcore --version` debería indicar 1.0.0-rc.5. Los comandos de la CLI funcionan mejor en una terminal, así que mantén una abierta junto a este cuaderno.

Hoy también necesitas las carpetas `runtime` y `agentcore` de `setup/setup_week3day4`. Si estás en la rama `week3day4-start`, ya están en la raíz del repositorio. En caso contrario, muévelas allí. En una terminal dentro de la carpeta `undercover`, ejecuta:

`git mv setup/setup_week3day4/runtime runtime`
`git mv setup/setup_week3day4/agentcore agentcore`

Runtime solo cobra mientras una sesión está trabajando, así que este laboratorio cuesta una fracción de un centavo en AWS, y las llamadas al modelo cuestan una fracción de centavo en OpenRouter. El primer despliegue prepara tu cuenta para la CLI, lo que incluye una clave KMS por $1 al mes, aproximadamente 3 centavos al día. Permanece para el Lab 2, que muestra cómo quitarla. La Parte 7 elimina los agentes.

Si estás en la ruta gratuita, este laboratorio necesita AWS, así que por favor sígueme. El Lab 2 tiene una guía para ejecutar el agente en Render.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Watching your AWS spend</h2>
            <span style="color:#118AB2;">Before you start, open a terminal in the <code>undercover</code> folder and run <code>uv run monitor.py</code>. It opens a page in your browser with what your AWS account has spent today and over the last 30 days, with and without your credits. AWS updates the figures up to three times a day.<br/><br/>For the official source of truth, please review Billing and Cost Management in your AWS console. Always have budgets with billing alerts, like the one from Day 1.
            </span>
        </td>
    </tr>
</table>

## Traducción al español

<h2 style="color:#118AB2;">Controlando tu gasto en AWS</h2>

Antes de empezar, abre una terminal en la carpeta `undercover` y ejecuta `uv run monitor.py`. Abre una página en tu navegador con lo que tu cuenta de AWS ha gastado hoy y en los últimos 30 días, con y sin tus créditos. AWS actualiza las cifras hasta tres veces al día.

Para la fuente oficial de verdad, revisa Billing and Cost Management en tu consola de AWS. Mantén siempre presupuestos con alertas de facturación, como el del Día 1.

In [ ]:
import json
import os
import time
import uuid
from pathlib import Path

import boto3
import httpx
import psycopg
from dotenv import load_dotenv, set_key

## Connect to AWS

boto3 uses the access key that `aws configure` saved on Day 1, in us-east-1.

## Traducción al español

## Conexión a AWS

`boto3` usa la clave de acceso que `aws configure` guardó el Día 1, en us-east-1.

In [ ]:
load_dotenv(override=True)

session = boto3.Session()
region = session.region_name
identity = session.client("sts").get_caller_identity()
account = identity["Account"]

print(f"Working as {identity['Arn']} in {region}")

In [ ]:
agentcore = session.client("bedrock-agentcore-control")
agentcore_runtime = session.client("bedrock-agentcore")
cloudformation = session.client("cloudformation")
logs = session.client("logs")

In [ ]:
for name in ["OPENROUTER_API_KEY", "DATABASE_URL"]:
    print(f"{name}: {'found' if os.getenv(name) else 'missing, please add it to .env'}")

## Part 1: Hello, AgentCore

The AgentCore CLI works with a project that holds an agent's code and its configuration. It runs the agent on your laptop and deploys it to AWS. `agentcore create` makes a project from a template, and we start with the smallest: `agent-python-minimal`, with no agent framework and no model. Its agent replies "Hello, world!".

The CLI won't make a project inside another one. The `undercover` repo already holds UnderCover's own AgentCore project, in its `agentcore` folder, which Lab 2 deploys. So we make the hello world beside the repo, in the folder that holds it. In your terminal, in the `undercover` folder, go up a level and make the project:

`cd ..`

`agentcore create --name hellolab --template agent-python-minimal`

It takes about 10 seconds, while the CLI installs the project's packages and starts a git repository for it. Open the `hellolab` folder in your editor. `agentcore/agentcore.json` describes everything the project deploys: one runtime, called `agent`, built from the code in `app/agent`. In `app/agent/main.py`, Runtime calls the function marked `@app.entrypoint` with each request's JSON, and returns what it returns.

## Traducción al español

## Parte 1: Hola, AgentCore

La CLI de AgentCore trabaja con un proyecto que contiene el código y la configuración del agente. Ejecuta el agente en tu portátil y lo despliega en AWS. `agentcore create` crea un proyecto a partir de una plantilla, y empezamos con la más pequeña: `agent-python-minimal`, sin framework de agente ni modelo. Su agente responde: "¡Hola, mundo!".

La CLI no creará un proyecto dentro de otro. El repositorio `undercover` ya contiene el proyecto de AgentCore propio de UnderCover, en la carpeta `agentcore`, que el Lab 2 despliega. Así que creamos el ejemplo hola mundo junto al repositorio, en la carpeta que lo contiene. En tu terminal, dentro de la carpeta `undercover`, sube un nivel y crea el proyecto:

`cd ..`

`agentcore create --name hellolab --template agent-python-minimal`

Tarda unos 10 segundos, mientras la CLI instala los paquetes del proyecto y crea un repositorio git para él. Abre la carpeta `hellolab` en tu editor. `agentcore/agentcore.json` describe todo lo que despliega el proyecto: un tiempo de ejecución llamado `agent`, construido desde el código en `app/agent`. En `app/agent/main.py`, Runtime llama a la función marcada con `@app.entrypoint` con el JSON de cada solicitud y devuelve lo que devuelve.

### Deploying it

`agentcore deploy` builds a zip of the code, with its packages for Linux on ARM, and deploys it with CloudFormation, with no Docker. The first deploy in an account creates:

- **The CDK bootstrap**, a CloudFormation stack called `CDKToolkit`, once for each account and region: an S3 bucket for the code, a KMS key for the bucket at $1 a month, and five IAM roles. One of them is the role CloudFormation deploys with, which has administrator access in your account. That's the CDK's default, and it's the largest change this course makes to your account. Lab 2 shows how to remove it.
- **A CloudFormation stack** called `AgentCore-hellolab-default`, with the runtime and the IAM role it runs as.
- **CloudWatch's Transaction Search**, on an account that doesn't have it yet, so you can see traces of an agent's work.

In your terminal, go into the project and deploy it:

`cd hellolab`

`agentcore deploy`

This first deploy takes about 4 minutes. Then call the agent on Runtime. `--payload` is the request's JSON, which this agent ignores:

`agentcore invoke --runtime agent --payload '{}'`

It answers `{"message": "Hello, world!"}`, followed by a line about the call.

## Traducción al español

### Despliegue

`agentcore deploy` construye un zip del código, con sus paquetes para Linux en ARM, y lo despliega con CloudFormation, sin Docker. El primer despliegue en una cuenta crea:

- **El bootstrap de CDK**, una pila de CloudFormation llamada `CDKToolkit`, una vez por cada cuenta y región: un bucket S3 para el código, una clave KMS para el bucket a $1 al mes y cinco roles IAM. Uno de ellos es el rol con el que CloudFormation despliega, que tiene acceso de administrador en tu cuenta. Ese es el valor predeterminado del CDK y es el cambio más grande que este curso hace en tu cuenta. El Lab 2 muestra cómo eliminarlo.
- **Una pila de CloudFormation** llamada `AgentCore-hellolab-default`, con el runtime y el rol IAM con el que se ejecuta.
- **Transaction Search de CloudWatch**, en una cuenta que aún no lo tiene, para que puedas ver trazas del trabajo de un agente.

En tu terminal, entra al proyecto y desplieguealo:

`cd hellolab`

`agentcore deploy`

Este primer despliegue tarda unos 4 minutos. Luego llama al agente en Runtime. `--payload` es el JSON de la solicitud, que este agente ignora:

`agentcore invoke --runtime agent --payload '{}'`

Responde `{"message": "Hello, world!"}`, seguida de una línea sobre la llamada.

### Removing it

`agentcore remove all` takes everything out of the project's configuration, and the next `agentcore deploy` deletes it from AWS. The bootstrap stays for the next deploy, and Part 7 deletes the runtime's log group.

`agentcore remove all --yes`

`agentcore deploy --yes`

Then you can delete the `hellolab` folder.

## Traducción al español

### Eliminación

`agentcore remove all` elimina todo de la configuración del proyecto y el siguiente `agentcore deploy` lo borra de AWS. El bootstrap permanece para el siguiente despliegue y la Parte 7 elimina el grupo de logs del runtime.

`agentcore remove all --yes`

`agentcore deploy --yes`

Entonces puedes eliminar la carpeta `hellolab`.

## Part 2: A LangGraph agent's project

The rest of the lab uses the project in `labs/undercoverlab`, with a small LangGraph agent. Please open that folder in your editor.

I made it with `agentcore create --name undercoverlab --template agent-python-langchain`. That template is a LangChain agent, and I changed it for UnderCover.

- `agentcore/agentcore.json` describes everything the project deploys
- `agentcore/cdk/` is the CDK project the CLI deploys with
- `app/agent/` holds the agent's code, with its own `pyproject.toml`

## Traducción al español

## Parte 2: Proyecto de un agente LangGraph

El resto del laboratorio usa el proyecto de `labs/undercoverlab`, con un agente pequeño de LangGraph. Abre esa carpeta en tu editor.

Lo creé con `agentcore create --name undercoverlab --template agent-python-langchain`. Esa plantilla es un agente de LangChain y yo la cambié para UnderCover.

- `agentcore/agentcore.json` describe todo lo que despliega el proyecto
- `agentcore/cdk/` es el proyecto CDK con el que la CLI despliega
- `app/agent/` contiene el código del agente, con su propio `pyproject.toml`

### The configuration

`agentcore/agentcore.json` has one runtime, called `agent`, built as a zip of the code in `app/agent`, with `main.py` as its entry point. I changed four things from the template. The runtime uses Python 3.12, as UnderCover does. The template's model is on Bedrock, and ours comes from OpenRouter, so the runtime names no model provider. A session ends after 60 idle seconds instead of 15 minutes. And the project has the `project=undercover` tag, like everything else we make on AWS.

The configuration also lists two credentials, `OpenRouter` and `DatabaseUrl`, for the agent's two secrets. In a project of your own, `agentcore add credentials` adds one.

The runtime uses Runtime's platform version V2, which starts each new session from a snapshot taken after the code is imported, so a session starts in about 2 seconds. The CLI has no setting for it yet, so I added a few lines to `agentcore/cdk/lib/cdk-stack.ts`, the CDK code the CLI deploys with.

## Traducción al español

### La configuración

`agentcore/agentcore.json` tiene un runtime llamado `agent`, construido como un zip del código en `app/agent`, con `main.py` como punto de entrada. Cambié cuatro cosas respecto a la plantilla. El runtime usa Python 3.12, como UnderCover. La plantilla del modelo está en Bedrock y la nuestra viene de OpenRouter, por lo que el runtime no nombra ningún proveedor de modelos. Una sesión termina tras 60 segundos inactivos en lugar de 15 minutos. Y el proyecto tiene la etiqueta `project=undercover`, como todo lo demás que creamos en AWS.

La configuración también lista dos credenciales, `OpenRouter` y `DatabaseUrl`, para los dos secretos del agente. En un proyecto propio, `agentcore add credentials` añade una.

El runtime usa la versión de plataforma V2 de Runtime, que inicia cada nueva sesión desde una instantánea tomada después de importar el código, por lo que una sesión empieza en unos 2 segundos. La CLI todavía no tiene un ajuste para ello, así que añadí unas cuantas líneas a `agentcore/cdk/lib/cdk-stack.ts`, el código CDK con el que despliega la CLI.

### The agent's code

`app/agent/main.py` gives the agent its jobs. `chat` is the template's agent with its one tool. `start` and `resume` work a claim with a small graph from Week 2: it proposes a payout, pauses for a handler with `interrupt()`, and decides again with the handler's note. The graph saves its progress in Supabase with `PostgresSaver`, in the `lab` schema from Week 2, so any session can resume it.

The model and the graph are made on the first request rather than when the code is imported. Every session starts from the snapshot taken after the import, so anything made at import would be shared by every session, including connections that are no longer open.

`app/agent/model/load.py` calls the model through OpenRouter instead of Bedrock, as UnderCover does. Its `secret` fetches one of the project's credentials: from AgentCore Identity when the agent is deployed, and from `agentcore/.env.local` when `agentcore dev` runs it on your laptop. The CLI gives the deployed agent each credential's name in Identity as an environment variable.

`app/agent/pyproject.toml` lists the agent's packages. I swapped the template's Bedrock package for OpenAI's, which works with OpenRouter, and added the Postgres checkpointer, at the versions UnderCover uses. The CLI installs them when it runs the agent and when it deploys it.

## Traducción al español

### El código del agente

`app/agent/main.py` da al agente sus tareas. `chat` es el agente de la plantilla con su única herramienta. `start` y `resume` trabajan con una reclamación usando un grafo pequeño de la Semana 2: propone un pago, se pausa para un manejador con `interrupt()` y decide de nuevo con la nota del manejador. El grafo guarda su progreso en Supabase con `PostgresSaver` en el esquema `lab` de la Semana 2, para que cualquier sesión pueda reanudarlo.

El modelo y el grafo se crean en la primera solicitud en lugar de cuando se importa el código. Cada sesión empieza desde la instantánea realizada después de la importación, así que cualquier cosa creada en la importación se compartiría entre todas las sesiones, incluidas conexiones que ya no están abiertas.

`app/agent/model/load.py` llama al modelo a través de OpenRouter en lugar de Bedrock, como hace UnderCover. Su `secret` recupera una de las credenciales del proyecto: desde AgentCore Identity cuando el agente está desplegado y desde `agentcore/.env.local` cuando `agentcore dev` lo ejecuta en tu portátil. La CLI da al agente desplegado el nombre de cada credencial en Identity como una variable de entorno.

`app/agent/pyproject.toml` lista los paquetes del agente. Cambié el paquete de Bedrock de la plantilla por el de OpenAI, que funciona con OpenRouter, y añadí el checkpointer de Postgres en las versiones que usa UnderCover. La CLI los instala cuando ejecuta el agente y cuando lo despliega.

## Part 3: The agent on your laptop

### The two secrets

The CLI reads each credential's value from `agentcore/.env.local`, which git ignores. `agentcore dev` gives the values to the agent on your laptop, and the first deploy stores them in AgentCore Identity, where the deployed agent fetches them. Each value's name is `AGENTCORE_CREDENTIAL_` and the credential's name in capitals. We copy the values from your `.env`, so they never appear on screen.

Runtime runs each session in a microVM of its own, with its own connections to the database, so the agent uses Supabase's transaction pooler on port 6543, as the app on Lambda does. It lends a connection for each transaction, so many sessions can share a few connections.

## Traducción al español

## Parte 3: El agente en tu portátil

### Las dos credenciales

La CLI lee el valor de cada credencial desde `agentcore/.env.local`, que git ignora. `agentcore dev` da los valores al agente en tu portátil, y el primer despliegue los almacena en AgentCore Identity, donde el agente desplegado los recupera. El nombre de cada valor es `AGENTCORE_CREDENTIAL_` y el nombre de la credencial en mayúsculas. Copiamos los valores desde tu `.env`, así que nunca aparecen en pantalla.

Runtime ejecuta cada sesión en su propia microVM, con sus propias conexiones a la base de datos, por lo que el agente usa el pooler de transacciones de Supabase en el puerto 6543, como hace la aplicación en Lambda. Presta una conexión para cada transacción, así que muchas sesiones pueden compartir unas pocas conexiones.

In [ ]:
LOCAL_ENV = Path("undercoverlab/agentcore/.env.local")
LOCAL_ENV.touch()

secrets = {
    "AGENTCORE_CREDENTIAL_OPENROUTER": os.environ["OPENROUTER_API_KEY"],
    "AGENTCORE_CREDENTIAL_DATABASEURL": os.environ["DATABASE_URL"].replace(":5432/", ":6543/"),
}

for name, value in secrets.items():
    set_key(LOCAL_ENV, name, value)

print(f"Wrote {', '.join(secrets)} to {LOCAL_ENV}")

### Running it on your laptop

`agentcore dev` runs the agent on your laptop as Runtime will run it: a web server on port 8080 with two routes, `/ping` to say it's healthy and `/invocations` to take a request.

In your terminal, go to the project's folder in the repo, where you'll run the rest of the lab's commands, and start it:

`cd ../undercover/labs/undercoverlab`

`agentcore dev`

The first time, uv installs the agent's packages, which takes about half a minute. When it says the server is running, carry on here.

## Traducción al español

### Ejecutarlo en tu portátil

`agentcore dev` ejecuta el agente en tu portátil como lo hará Runtime: un servidor web en el puerto 8080 con dos rutas, `/ping` para indicar que está bien y `/invocations` para recibir una solicitud.

En tu terminal, ve a la carpeta del proyecto dentro del repositorio, donde ejecutarás el resto de comandos del laboratorio y arráncalo:

`cd ../undercover/labs/undercoverlab`

`agentcore dev`

La primera vez, `uv` instala los paquetes del agente, lo que tarda unos 30 segundos. Cuando diga que el servidor está funcionando, continúa aquí.

In [ ]:
print(httpx.get("http://127.0.0.1:8080/ping").json())

In [ ]:
question = {"action": "chat", "prompt": "What is 1,180 minus 240?"}

reply = httpx.post("http://127.0.0.1:8080/invocations", json=question, timeout=120)
print(reply.json())

It works on your laptop. Stop `agentcore dev` with Ctrl+C in its terminal.

## Part 4: Deploying to Runtime

This deploy creates the two credentials in AgentCore Identity, and a stack called `AgentCore-undercoverlab-default` with the runtime and its role. The bootstrap from Part 1 is already there.

The CLI deploys with the CDK project in `agentcore/cdk`, and `agentcore create` installed its packages for the hello world. This project came from the repo, so install them once. In your terminal, in `undercoverlab`, run:

`npm ci --prefix agentcore/cdk`

`agentcore deploy`

It takes about 7 minutes, most of it while Runtime makes the snapshot. When it's done, `agentcore status` lists what the project has deployed.

`agentcore invoke` calls the deployed agent with a payload of JSON. This prints the command with our question, for you to copy into your terminal. The JSON goes in on standard input, with `--payload -`, because PowerShell on a PC drops the quotes from JSON passed as an argument.

## Traducción al español

Funciona en tu portátil. Para detener `agentcore dev`, pulsa Ctrl+C en su terminal.

## Parte 4: Despliegue en Runtime

Este despliegue crea las dos credenciales en AgentCore Identity y una pila llamada `AgentCore-undercoverlab-default` con el runtime y su rol. El bootstrap de la Parte 1 ya está allí.

La CLI despliega con el proyecto CDK en `agentcore/cdk`, y `agentcore create` instaló sus paquetes para el ejemplo hola mundo. Este proyecto proviene del repositorio, así que instálalos una sola vez. En tu terminal, dentro de `undercoverlab`, ejecuta:

`npm ci --prefix agentcore/cdk`

`agentcore deploy`

Tarda unos 7 minutos, la mayor parte mientras Runtime crea la instantánea. Cuando termine, `agentcore status` lista lo que el proyecto ha desplegado.

`agentcore invoke` llama al agente desplegado con una carga JSON. Esto imprime el comando con nuestra pregunta para que lo copies en tu terminal. El JSON entra por la entrada estándar con `--payload -`, porque PowerShell en un PC elimina las comillas del JSON pasadas como argumento.

In [ ]:
print(f"echo '{json.dumps(question)}' | agentcore invoke --runtime agent --payload -")

## Part 5: Sessions

### Cold and warm

Runtime runs each session in a microVM. You can specify a session id to create one; a session ends after 60 idle seconds, as we set in the configuration. From Python, `invoke_agent_runtime` calls the agent in the session we name, with an id of at least 33 characters.

## Traducción al español

## Parte 5: Sesiones

### Fría y cálida

Runtime ejecuta cada sesión en una microVM. Puedes especificar un identificador de sesión para crear una; una sesión termina después de 60 segundos inactivos, como configuramos. Desde Python, `invoke_agent_runtime` llama al agente en la sesión que nombramos, con un id de al menos 33 caracteres.

In [ ]:
def find_runtime():
    """The lab's runtime, found by its name."""
    runtimes = agentcore.list_agent_runtimes()["agentRuntimes"]
    return next(r for r in runtimes if r["agentRuntimeName"].startswith("undercoverlab"))


runtime = find_runtime()
runtime_arn = runtime["agentRuntimeArn"]
details = agentcore.get_agent_runtime(agentRuntimeId=runtime["agentRuntimeId"])
print(f"{runtime_arn} on platform version {details['platformVersion']}")

In [ ]:
def new_session():
    """A new session id, long enough for Runtime."""
    return f"undercoverlab-{uuid.uuid4()}"


def call(payload, session_id):
    """Call the agent in a session, and print its reply and how long it took."""
    started = time.time()
    response = agentcore_runtime.invoke_agent_runtime(agentRuntimeArn=runtime_arn, runtimeSessionId=session_id, payload=json.dumps(payload))
    reply = json.loads(response["response"].read())
    print(f"{time.time() - started:.1f} seconds: {reply}")
    return reply

A `hello` asks the agent to do nothing, so its time is how long Runtime takes. The first call starts a session: Runtime starts a microVM from the snapshot. The second goes to the same session, which is warm.

## Traducción al español

Una `hello` le pide al agente que no haga nada, así que su tiempo es el que tarda Runtime. La primera llamada inicia una sesión: Runtime arranca una microVM desde la instantánea. La segunda va a la misma sesión, que ya está caliente.

In [ ]:
hello = {"action": "hello"}
session_id = new_session()

reply = call(hello, session_id)
reply = call(hello, session_id)

## Part 6: Pausing in one session and resuming in another

The claim, CLM-9001, asks for $1,180 after a canceled flight, with two taxi receipts for $240 each. In the first session, the agent proposes a payout and the graph pauses for a handler. Each claim gets a thread of its own.

## Traducción al español

## Parte 6: Pausar en una sesión y reanudar en otra

La reclamación CLM-9001 solicita $1,180 tras un vuelo cancelado, con dos recibos de taxi por $240 cada uno. En la primera sesión, el agente propone un pago y el grafo se pausa para un manejador. Cada reclamación tiene su propio hilo.

In [ ]:
thread_id = str(uuid.uuid4())
first_session = new_session()

reply = call({"action": "start", "thread_id": thread_id}, first_session)

We end the first session, as Runtime would after 60 idle seconds.

## Traducción al español

La primera sesión se termina, como haría Runtime después de 60 segundos inactivos.

In [ ]:
response = agentcore_runtime.stop_runtime_session(agentRuntimeArn=runtime_arn, runtimeSessionId=first_session)
print(f"Stopped session {first_session}")

The session is gone, and the claim's place is saved in Supabase as checkpoints on its thread.

## Traducción al español

La sesión ha desaparecido y el lugar de la reclamación queda guardado en Supabase como checkpoints en su hilo.

In [ ]:
with psycopg.connect(os.environ["DATABASE_URL"]) as connection:
    rows = connection.execute("SELECT checkpoint_id FROM lab.checkpoints WHERE thread_id = %s", (thread_id,)).fetchall()

print(f"{len(rows)} checkpoints saved on thread {thread_id}")

Hours later, a handler sends the proposal back with a note. A new session, in a new microVM, loads the thread from Supabase and carries on from the pause.

## Traducción al español

Horas más tarde, un manejador devuelve la propuesta con una nota. Una nueva sesión, en una nueva microVM, carga el hilo desde Supabase y continúa desde la pausa.

In [ ]:
second_session = new_session()
note = {"action": "resume", "thread_id": thread_id, "note": "The taxi receipt is in twice."}

reply = call(note, second_session)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Where the state lives</h2>
            <span style="color:#118AB2;">The graph saves the state it in Supabase after every step, and the second session retrieved it by the thread's id. Lab 2 works the same way: UnderCover's agent saves every claim's progress in Supabase, and a claim sent back resumes in a new session.
            </span>
        </td>
    </tr>
</table>

## Traducción al español

<h2 style="color:#118AB2;">Dónde vive el estado</h2>

El grafo guarda el estado en Supabase después de cada paso, y la segunda sesión lo recuperó mediante el id del hilo. El Lab 2 funciona igual: el agente de UnderCover guarda el progreso de cada reclamación en Supabase, y una reclamación enviada de vuelta se reanuda en una nueva sesión.

## Part 7: Teardown

As with the hello world, `agentcore remove all` and the next deploy delete the project from AWS: the stack with the runtime and its role, and the two credentials. In your terminal, in `undercoverlab`, run:

`agentcore remove all --yes`

`agentcore deploy --yes`

The deploy takes about a minute. `remove all` emptied `agentcore/agentcore.json` too, so put it back from git, ready for another run:

`git restore agentcore/agentcore.json`

## Traducción al español

## Parte 7: Eliminación

Al igual que con el ejemplo hola mundo, `agentcore remove all` y el siguiente despliegue eliminan el proyecto de AWS: la pila con el runtime y su rol, y las dos credenciales. En tu terminal, dentro de `undercoverlab`, ejecuta:

`agentcore remove all --yes`

`agentcore deploy --yes`

El despliegue tarda unos minutos. `remove all` también vació `agentcore/agentcore.json`, así que vuelve a ponerlo desde git, listo para otra ejecución:

`git restore agentcore/agentcore.json`

The CLI leaves two things behind. Each runtime's log group, the hello world's and this one's, keeps its logs forever, so we delete them by name. The CDK bootstrap stays for Lab 2, which shows how to remove it at the end of the day.

## Traducción al español

La CLI deja dos cosas detrás. El grupo de logs de cada runtime, el del hola mundo y este, conserva los logs para siempre, así que los eliminamos por nombre. El bootstrap del CDK se queda para el Lab 2, que muestra cómo eliminarlo al final del día.

In [ ]:
LOG_GROUPS = ["/aws/bedrock-agentcore/runtimes/hellolab", "/aws/bedrock-agentcore/runtimes/undercoverlab"]

for prefix in LOG_GROUPS:
    for page in logs.get_paginator("describe_log_groups").paginate(logGroupNamePrefix=prefix):
        for group in page["logGroups"]:
            logs.delete_log_group(logGroupName=group["logGroupName"])
            print(f"Deleted log group {group['logGroupName']}")

In [ ]:
def show_resources():
    """Print what this lab created that still exists."""
    stacks = [s["StackName"] for page in cloudformation.get_paginator("describe_stacks").paginate() for s in page["Stacks"]]
    providers = agentcore.list_api_key_credential_providers()["credentialProviders"]
    projects = ("hellolab", "undercoverlab")
    found = {
        "Runtimes": [r["agentRuntimeName"] for r in agentcore.list_agent_runtimes()["agentRuntimes"] if r["agentRuntimeName"].startswith(projects)],
        "Credentials": [p["name"] for p in providers if p["name"].startswith(projects)],
        "Stacks": [name for name in stacks if name.startswith(("AgentCore-hellolab", "AgentCore-undercoverlab", "CDKToolkit"))],
    }
    for label, names in found.items():
        print(f"{label + ':':<14}{', '.join(names) or 'none'}")


show_resources()

Only `CDKToolkit`, the bootstrap, should be left.

Over the next day or two, keep an eye on your spend with `uv run monitor.py`. For the official figures, review Billing and Cost Management in your AWS console.

## Traducción al español

Solo debería quedar `CDKToolkit`, el bootstrap.

Durante el próximo día o dos, vigila tu gasto con `uv run monitor.py`. Para las cifras oficiales, revisa Billing and Cost Management en tu consola de AWS.